In [1]:
# ==========================================
#   IMPORT LIBRARIES & SETUP
# ==========================================
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'   # implicit recommends this for ALS

%load_ext autoreload
%autoreload 2

from metrics import evaluate
import pandas as pd 
import numpy as np
from scipy.sparse import csr_matrix
import implicit
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import CosineRecommender
import itertools
import time

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
articles=pd.read_parquet('data/articles_processed.parquet')
customers=pd.read_parquet('data/customers_processed.parquet')
transactions=pd.read_parquet('data/transactions_processed.parquet')


In [3]:
#=================================================================================
#                        MAPPING
#=================================================================================

unique_customers = customers['customer_id'].unique()
unique_articles = articles['article_id'].unique()

# Mappings
customer2idx = {c: i for i, c in enumerate(unique_customers)}
article2idx = {a: i for i, a in enumerate(unique_articles)}

# Reverse Mappings 
idx2customer = {i: c for c, i in customer2idx.items()}
idx2article = {i: a for a, i in article2idx.items()}

# Mapping transactions-ի վրա
transactions['user_idx'] = transactions['customer_id'].map(customer2idx)
transactions['item_idx'] = transactions['article_id'].map(article2idx)

# Check: no unmapped IDs
print(transactions['user_idx'].isna().sum(), transactions['item_idx'].isna().sum())

0 0


In [4]:
#================================================================================
#                  TRAIN-VALIDATION-TEST SPLIT
#================================================================================
last_date = transactions['t_dat'].max()
test_start = last_date - pd.Timedelta(days=6)    # last week
val_start = test_start - pd.Timedelta(days=7)    # the week before last

train = transactions[transactions['t_dat'] < val_start]
val = transactions[(transactions['t_dat'] >= val_start) & (transactions['t_dat'] < test_start)]
test = transactions[transactions['t_dat'] >= test_start]

train = pd.concat([train, val])

In [5]:
#=================================================================================
#                              CSR Matrix
#=================================================================================

train_user_item = train.groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')

rows = train_user_item['user_idx'].values
cols = train_user_item['item_idx'].values
weights = train_user_item['weight'].values.astype(np.float32)

n_users = len(customer2idx)
n_items = len(article2idx)


user_item_matrix = csr_matrix((weights, (rows, cols)), shape=(n_users, n_items))


In [6]:

recent_popular = train[train['t_dat'] >= train['t_dat'].max() - pd.Timedelta(days=6)]
fallback_top12 = recent_popular.groupby('article_id').size().nlargest(12).index.tolist()

def predict_implicit_model(model, val_df, k=12):
    
    actuals_dict = val_df.groupby('customer_id')['article_id'].unique().apply(set).to_dict()
    val_customers = list(actuals_dict.keys())

    user_idxs = np.array([customer2idx[c_id] for c_id in val_customers])

    has_history = np.asarray(user_item_matrix[user_idxs].getnnz(axis=1)) > 0

    ids, scores = model.recommend(
        user_idxs,
        user_item_matrix[user_idxs],
        N=k,
        filter_already_liked_items=False
    )

    predictions = {}
    for i, c_id in enumerate(val_customers):
        if has_history[i]:
            recs = [idx2article[int(idx)] for idx in ids[i] if idx >= 0]
        else:
            recs = []  # cold customer
        fill = [p for p in fallback_top12 if p not in recs]
        predictions[c_id] = (recs + fill)[:k]

    return predictions, actuals_dict

In [7]:
# ================================================================================
#                       RESULTS DICTIONARY INITIALIZATION
# ================================================================================

results = {
    'Model': [],
    'Description': [],
    'MAP@12': [],
    'NDCG@12': [],
    'Precision@12': [],
    'Recall@12': [],
    'Novelty@12': [],
    'Coverage': []
}

def log_results(model_name, description, metrics_dict):
    results['Model'].append(model_name)
    results['Description'].append(description)
    results['MAP@12'].append(metrics_dict['MAP@12'])
    results['NDCG@12'].append(metrics_dict['NDCG@12'])
    results['Precision@12'].append(metrics_dict['Precision@12'])
    results['Recall@12'].append(metrics_dict['Recall@12'])
    results['Novelty@12'].append(metrics_dict['Novelty@12'])
    results['Coverage'].append(metrics_dict['Coverage'])
    print(f"✅ Saved results for: {model_name}")

In [8]:
# Մոդելի աշխատանքը գնահատելու համար

total_catalog_items = articles['article_id'].nunique()
sales_series = train['article_id'].value_counts()

total_sales = len(train)

novelty_dict = (
    (-np.log2(sales_series / total_sales)).to_dict()
)

In [9]:
# ================================================================================
#                       BASELINE 1: POPULARITY BASELINE
# ================================================================================


top12_popular = train['article_id'].value_counts().head(12).index.tolist()
actuals = test.groupby('customer_id')['article_id'].unique().to_dict() 

predictions = {user: top12_popular for user in actuals.keys()}

pop_results = evaluate(
    predictions=predictions,
    actuals=actuals,
    total_catalog_items=total_catalog_items,
    novelty_dict=novelty_dict,
    k=12
)

log_results("Popularity Baseline", "Top 12 global popular items", pop_results)

print("=== GLOBAL POPULARITY BASELINE RESULTS ===")
for metric, value in pop_results.items():
    print(f"{metric:<15}: {value:.6f}")

✅ Saved results for: Popularity Baseline
=== GLOBAL POPULARITY BASELINE RESULTS ===
Precision@12   : 0.001975
Recall@12      : 0.007749
MAP@12         : 0.002899
NDCG@12        : 0.005311
Novelty@12     : 10.218052
Coverage       : 0.000114


In [10]:
#=================================================================================
#                       Recent Popularity Baseline
#=================================================================================

end = train['t_dat'].max()
start = end - pd.Timedelta(days=6)

top12_recent = (
    train[train['t_dat'] >= start]
    .groupby('article_id')
    .size()
    .nlargest(12)
    .index.tolist()
)

actuals = test.groupby('customer_id')['article_id'].unique().to_dict()  
actuals = {user: set(items) for user, items in actuals.items()}


predictions = {user: top12_recent for user in actuals.keys()}

recent_pop_results = evaluate(
    predictions=predictions,
    actuals=actuals,
    total_catalog_items=total_catalog_items,
    novelty_dict=novelty_dict,
    k=12
)


log_results("Recent Popularity Baseline", "Top 12 items from last 7 days of train", recent_pop_results)

print("=== RECENT (LAST 7 DAYS) POPULARITY BASELINE RESULTS ===")
for metric, value in recent_pop_results.items():
    print(f"{metric:<15}: {value:.6f}")

✅ Saved results for: Recent Popularity Baseline
=== RECENT (LAST 7 DAYS) POPULARITY BASELINE RESULTS ===
Precision@12   : 0.005907
Recall@12      : 0.025504
MAP@12         : 0.008748
NDCG@12        : 0.015757
Novelty@12     : 13.944722
Coverage       : 0.000114


In [11]:
#=================================================================================
#                BASELINE 3: Repeat Purchases + Recent Popularity fallback
#=================================================================================


customer_article_counts = (
    train.groupby(['customer_id', 'article_id'])
    .size()
    .reset_index(name='count')
    .sort_values(['customer_id', 'count'], ascending=[True, False])
)

customer_history = customer_article_counts.groupby('customer_id')['article_id'].apply(list).to_dict()

def predict_user(c_id):
    own = customer_history.get(c_id, [])
    if len(own) < 12:
        fill = [p for p in top12_recent if p not in own]
        own = own + fill[:12 - len(own)]
    return own[:12]

actuals = test.groupby('customer_id')['article_id'].unique().to_dict()
actuals = {user: set(items) for user, items in actuals.items()}

predictions = {c: predict_user(c) for c in actuals.keys()}

repeat_pop_results = evaluate(
    predictions=predictions,
    actuals=actuals,
    total_catalog_items=total_catalog_items,
    novelty_dict=novelty_dict,
    k=12
)


log_results("Repeat Purchases + Recent Pop Baseline", "Customer history filled with recent popular items", repeat_pop_results)

print("=== REPEAT PURCHASES + RECENT POPULARITY BASELINE RESULTS ===")
for metric, value in repeat_pop_results.items():
    print(f"{metric:<15}: {value:.6f}")

✅ Saved results for: Repeat Purchases + Recent Pop Baseline
=== REPEAT PURCHASES + RECENT POPULARITY BASELINE RESULTS ===
Precision@12   : 0.004443
Recall@12      : 0.024587
MAP@12         : 0.009699
NDCG@12        : 0.015377
Novelty@12     : 14.460119
Coverage       : 0.563899


In [12]:
#==================================================================================
#                     BASELINE 4: Limited History + Recent Popularity
#==================================================================================

K_HISTORY = 5  

def predict_baseline4(customer_id):
    own = customer_history.get(customer_id, [])[:K_HISTORY]
    
    if len(own) < 12:
        fill = [p for p in top12_recent if p not in own]
        own = own + fill[:12 - len(own)]
        
    return own[:12]

actuals = test.groupby('customer_id')['article_id'].unique().to_dict()  
actuals = {user: set(items) for user, items in actuals.items()}

predictions = {c: predict_baseline4(c) for c in actuals.keys()}

b4_results = evaluate(
    predictions=predictions,
    actuals=actuals,
    total_catalog_items=total_catalog_items,
    novelty_dict=novelty_dict,
    k=12
)

log_results("Top 5 Repeat + Recent Pop Baseline", "Last 5 repeated items filled with recent popular", b4_results)

print("=== BASELINE 4 (TOP 5 REPEAT + RECENT POPULARITY) RESULTS ===")
for metric, value in b4_results.items():
    print(f"{metric:<15}: {value:.6f}")

✅ Saved results for: Top 5 Repeat + Recent Pop Baseline
=== BASELINE 4 (TOP 5 REPEAT + RECENT POPULARITY) RESULTS ===
Precision@12   : 0.006138
Recall@12      : 0.031434
MAP@12         : 0.010562
NDCG@12        : 0.018309
Novelty@12     : 13.992026
Coverage       : 0.423869


In [13]:
#=================================================================================
#                   Item-Item Cosine Similarity
#=================================================================================

cosine_model = CosineRecommender(K=20)
cosine_model.fit(user_item_matrix)
print("Cosine Similarity model successfully fitted!")
print(cosine_model.similarity.shape)

cosine_preds, actuals_dict = predict_implicit_model(cosine_model, test, k=12)

cosine_results = evaluate(cosine_preds, actuals_dict, total_catalog_items,novelty_dict, k=12)

log_results("Item-Item Cosine Similarity", "Cosine Similarity with K=20", cosine_results)

print("--- Cosine Similarity Results ---")
for metric_name, value in cosine_results.items():
    print(f"{metric_name}: {value:.5f}")


C:\Users\user\AppData\Roaming\Python\Python314\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.16939878463745117 seconds
  warnings.warn(
100%|██████████| 105542/105542 [00:07<00:00, 13930.40it/s]


Cosine Similarity model successfully fitted!
(105542, 105542)
✅ Saved results for: Item-Item Cosine Similarity
--- Cosine Similarity Results ---
Precision@12: 0.00490
Recall@12: 0.02707
MAP@12: 0.01092
NDCG@12: 0.01704
Novelty@12: 14.76695
Coverage: 0.68818


In [16]:
# =================================================================================
#           BEST HYBRID: 6 RECENT REPEAT PURCHASES + COSINE SIMILARITY
# =================================================================================

cutoff = train['t_dat'].max() - pd.Timedelta(days=27)
recent_train = train[train['t_dat'] >= cutoff]

recent_ui = recent_train.groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')
recent_ui = recent_ui.sort_values(['user_idx', 'weight'], ascending=[True, False])


cosine_preds, actuals_dict = predict_implicit_model(cosine_model, test, k=12) 


BEST_N_REPEAT = 6

top_repeat = recent_ui.groupby('user_idx').head(BEST_N_REPEAT)
repeat_by_user = top_repeat.groupby('user_idx')['item_idx'].apply(list).to_dict()

hybrid_preds = {}
for c_id, model_recs in cosine_preds.items():
    repeat = [idx2article[int(i)] for i in repeat_by_user.get(customer2idx[c_id], [])]
    fill = [r for r in model_recs if r not in repeat]   
    hybrid_preds[c_id] = (repeat + fill)[:12]

best_hybrid_results = evaluate(
    predictions=hybrid_preds, 
    actuals=actuals_dict, 
    total_catalog_items=total_catalog_items, 
    novelty_dict=novelty_dict, 
    k=12
)

log_results(
    model_name="Hybrid: 6 recent repeat + Cosine", 
    description="Top 6 repeat items from last 4 weeks filled with Cosine recs", 
    metrics_dict=best_hybrid_results
)

print("=== BEST HYBRID (6 RECENT REPEAT + COSINE) TEST RESULTS ===")
for metric_name, value in best_hybrid_results.items():
    print(f"{metric_name:<15}: {value:.6f}")

✅ Saved results for: Hybrid: 6 recent repeat + Cosine
=== BEST HYBRID (6 RECENT REPEAT + COSINE) TEST RESULTS ===
Precision@12   : 0.007317
Recall@12      : 0.040689
MAP@12         : 0.021635
NDCG@12        : 0.030101
Novelty@12     : 14.851615
Coverage       : 0.675238


In [17]:
# ================================================================================
#                               BEST ALS MODEL
# ================================================================================

als_matrix = user_item_matrix.copy()
als_matrix.data = 1 + np.log(als_matrix.data)

BEST_FACTORS = 64
BEST_REG = 0.1     
BEST_ALPHA = 5     

best_als_model = AlternatingLeastSquares(
    factors=BEST_FACTORS, 
    regularization=BEST_REG, 
    alpha=BEST_ALPHA,
    iterations=15, 
    random_state=42
)
best_als_model.fit(als_matrix, show_progress=True)


als_preds, actuals_dict = predict_implicit_model(best_als_model, test, k=12)

als_results = evaluate(
    predictions=als_preds, 
    actuals=actuals_dict, 
    total_catalog_items=total_catalog_items, 
    novelty_dict=novelty_dict, 
    k=12
)


log_results(
    model_name=f"Standard ALS (f={BEST_FACTORS}, reg={BEST_REG}, alpha={BEST_ALPHA})", 
    description="Implicit ALS Matrix Factorization on full interaction matrix", 
    metrics_dict=als_results
)

print("\n=== STANDARD ALS (TEST RESULTS) ===")
for metric_name, value in als_results.items():
    print(f"{metric_name:<15}: {value:.6f}")

100%|██████████| 15/15 [01:47<00:00,  7.15s/it]


✅ Saved results for: Standard ALS (f=64, reg=0.1, alpha=5)

=== STANDARD ALS (TEST RESULTS) ===
Precision@12   : 0.004512
Recall@12      : 0.020454
MAP@12         : 0.008401
NDCG@12        : 0.013906
Novelty@12     : 12.035216
Coverage       : 0.018031


In [18]:
# ==================================================================================
#                     BEST ALS WITH TIME-DECAY WEIGHTS
# ==================================================================================


BEST_HALF_LIFE = 14 


age_days = (train['t_dat'].max() - train['t_dat']).dt.days 
w = 0.5 ** (age_days / BEST_HALF_LIFE)

ui = (train.assign(w=w)
    .groupby(['user_idx', 'item_idx'])['w'].sum()
    .reset_index())

decay_matrix = csr_matrix(
    (ui['w'].values.astype(np.float32), (ui['user_idx'].values, ui['item_idx'].values)),
    shape=(n_users, n_items)
)


decay_model = AlternatingLeastSquares(
    factors=64, 
    regularization=0.01, 
    alpha=5,
    iterations=15, 
    random_state=42
)
decay_model.fit(decay_matrix, show_progress=True)


decay_preds, actuals_dict = predict_implicit_model(decay_model, test, k=12)

time_decay_results = evaluate(
    predictions=decay_preds, 
    actuals=actuals_dict, 
    total_catalog_items=total_catalog_items, 
    novelty_dict=novelty_dict, 
    k=12
)


log_results(
    model_name=f"ALS Time-Decay (half_life={BEST_HALF_LIFE})", 
    description=f"ALS with exponential time-decay weights (half_life={BEST_HALF_LIFE} days)", 
    metrics_dict=time_decay_results
)

print(f"=== TIME-DECAY ALS (HALF_LIFE={BEST_HALF_LIFE}) TEST RESULTS ===")
for metric_name, value in time_decay_results.items():
    print(f"{metric_name:<15}: {value:.6f}")

100%|██████████| 15/15 [02:06<00:00,  8.46s/it]


✅ Saved results for: ALS Time-Decay (half_life=14)
=== TIME-DECAY ALS (HALF_LIFE=14) TEST RESULTS ===
Precision@12   : 0.008891
Recall@12      : 0.043477
MAP@12         : 0.018751
NDCG@12        : 0.029321
Novelty@12     : 13.754959
Coverage       : 0.009001


In [19]:
# ===================================================================================
#      BEST HYBRID: 6 RECENT REPEAT PURCHASES + TIME-DECAY ALS (half_life=14)
# ===================================================================================

# 1. Train Time-Decay ALS (half_life=14)
age_days = (train['t_dat'].max() - train['t_dat']).dt.days
w = 0.5 ** (age_days / 14)
ui = train.assign(w=w).groupby(['user_idx', 'item_idx'])['w'].sum().reset_index()

decay_matrix = csr_matrix(
    (ui['w'].values.astype(np.float32), (ui['user_idx'].values, ui['item_idx'].values)),
    shape=(n_users, n_items)
)

als_decay = AlternatingLeastSquares(
    factors=64, 
    regularization=0.01, 
    alpha=5,
    iterations=15, 
    random_state=42
)
als_decay.fit(decay_matrix, show_progress=True)


als_preds, actuals_dict = predict_implicit_model(als_decay, test, k=12)


BEST_N_REPEAT = 6  

top_repeat = recent_ui.groupby('user_idx').head(BEST_N_REPEAT)
repeat_by_user = top_repeat.groupby('user_idx')['item_idx'].apply(list).to_dict()

hybrid_preds = {}
for c_id, model_recs in als_preds.items():
    repeat = [idx2article[int(i)] for i in repeat_by_user.get(customer2idx[c_id], [])]
    fill = [r for r in model_recs if r not in repeat]
    hybrid_preds[c_id] = (repeat + fill)[:12]


hybrid_als_results = evaluate(
    predictions=hybrid_preds, 
    actuals=actuals_dict, 
    total_catalog_items=total_catalog_items, 
    novelty_dict=novelty_dict, 
    k=12
)


log_results(
    model_name=f"Hybrid: {BEST_N_REPEAT} recent repeat + Time-Decay ALS", 
    description=f"Top {BEST_N_REPEAT} repeat items filled with Time-Decay ALS (half_life=14)", 
    metrics_dict=hybrid_als_results
)

print(f"=== HYBRID ({BEST_N_REPEAT} RECENT REPEAT + TIME-DECAY ALS) TEST RESULTS ===")
for metric_name, value in hybrid_als_results.items():
    print(f"{metric_name:<15}: {value:.6f}")

100%|██████████| 15/15 [01:46<00:00,  7.08s/it]


✅ Saved results for: Hybrid: 6 recent repeat + Time-Decay ALS
=== HYBRID (6 RECENT REPEAT + TIME-DECAY ALS) TEST RESULTS ===
Precision@12   : 0.010731
Recall@12      : 0.055419
MAP@12         : 0.025098
NDCG@12        : 0.037542
Novelty@12     : 13.931037
Coverage       : 0.154602


In [20]:
# ===================================================================================
#     BEST REPEAT + ALS (OPTIMAL HISTORY WINDOW & N_REPEAT)
# ===================================================================================


BEST_WEEKS = 2
BEST_N_REPEAT = 5

cutoff = train['t_dat'].max() - pd.Timedelta(days=7 * BEST_WEEKS - 1)
recent_ui = (train[train['t_dat'] >= cutoff]
            .groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')
            .sort_values(['user_idx', 'weight'], ascending=[True, False]))

als_preds, actuals_dict = predict_implicit_model(als_decay, test, k=12)


top_repeat = recent_ui.groupby('user_idx').head(BEST_N_REPEAT)
repeat_by_user = top_repeat.groupby('user_idx')['item_idx'].apply(list).to_dict()

hybrid_preds = {}
for c_id, model_recs in als_preds.items():
    repeat = [idx2article[int(i)] for i in repeat_by_user.get(customer2idx[c_id], [])]
    fill = [r for r in model_recs if r not in repeat]
    hybrid_preds[c_id] = (repeat + fill)[:12]


final_window_results = evaluate(
    predictions=hybrid_preds, 
    actuals=actuals_dict, 
    total_catalog_items=total_catalog_items, 
    novelty_dict=novelty_dict, 
    k=12
)


log_results(
    model_name=f"Hybrid: Repeat ({BEST_WEEKS}w window, N={BEST_N_REPEAT}) + ALS Decay",
    description=f"Optimal history window ({BEST_WEEKS} weeks) with {BEST_N_REPEAT} repeat items filled by Time-Decay ALS",
    metrics_dict=final_window_results
)

print(f"=== FINAL HYBRID (WINDOW={BEST_WEEKS}w, N={BEST_N_REPEAT}) TEST RESULTS ===")
for metric_name, value in final_window_results.items():
    print(f"{metric_name:<15}: {value:.6f}")

✅ Saved results for: Hybrid: Repeat (2w window, N=5) + ALS Decay
=== FINAL HYBRID (WINDOW=2w, N=5) TEST RESULTS ===
Precision@12   : 0.010674
Recall@12      : 0.054651
MAP@12         : 0.026059
NDCG@12        : 0.038243
Novelty@12     : 13.857842
Coverage       : 0.114201


In [26]:
os.makedirs('results', exist_ok=True)

test_results_df = pd.DataFrame(results)

metric_cols = ['MAP@12', 'NDCG@12', 'Precision@12', 'Recall@12', 'Novelty@12', 'Coverage']
for col in metric_cols:
    if col in test_results_df.columns:
        test_results_df[col] = test_results_df[col].astype(float).round(5)

output_path = 'results/model_test_results.csv'
test_results_df.to_csv(output_path)

test_results_df

,Model,Description,MAP@12,NDCG@12,Precision@12,Recall@12,Novelty@12,Coverage
0,Popularity Baseline,Top 12 global popular items,0.00290,0.00531,0.00198,0.00775,10.21805,0.00011
1,Recent Popularity Baseline,Top 12 items from last 7 days of train,0.00875,0.01576,0.00591,0.02550,13.94472,0.00011
2,Repeat Purchases + Recent Pop Baseline,Customer history filled with recent popular items,0.00970,0.01538,0.00444,0.02459,14.46012,0.56390
3,Top 5 Repeat + Recent Pop Baseline,Last 5 repeated items filled with recent popular,0.01056,0.01831,0.00614,0.03143,13.99203,0.42387
4,Item-Item Cosine Similarity,Cosine Similarity with K=20,0.01092,0.01704,0.00490,0.02707,14.76695,0.68818
5,Hybrid: 6 recent repeat + Cosine,Top 6 repeat items from last 4 weeks filled wi...,0.02163,0.03010,0.00732,0.04069,14.85162,0.67524
6,"Standard ALS (f=64, reg=0.1, alpha=5)",Implicit ALS Matrix Factorization on full inte...,0.00840,0.01391,0.00451,0.02045,12.03522,0.01803
7,ALS Time-Decay (half_life=14),ALS with exponential time-decay weights (half_...,0.01875,0.02932,0.00889,0.04348,13.75496,0.00900
8,Hybrid: 6 recent repeat + Time-Decay ALS,Top 6 repeat items filled with Time-Decay ALS ...,0.02510,0.03754,0.01073,0.05542,13.93104,0.15460
9,"Hybrid: Repeat (2w window, N=5) + ALS Decay",Optimal history window (2 weeks) with 5 repeat...,0.02606,0.03824,0.01067,0.05465,13.85784,0.11420
